# 🐣 Airflow, lo más básico: dos DAGs

Sin vueltas. Vamos a leer ** dos archivos** y explicar qué hace cada línea.

> Si entendés estos dos ejemplos, entendés lo esencial de Airflow.

## Primero: ¿qué es Airflow? (en 2 frases)

Airflow es un programa que **corre tus tareas en orden y a horario**, y te muestra en una pantalla web si salieron bien o mal.

Vos le decís *"hacé esto, después esto otro"* en un archivo de Python. Ese archivo se llama **DAG**.

> 🧠 **DAG = tu lista de pasos.** Nada más. La sigla significa "grafo dirigido acíclico", pero olvidate del nombre: pensalo como una **receta** (paso 1, paso 2, paso 3).

## Tres palabras que vas a ver todo el tiempo

- **DAG** = el archivo con tu lista de pasos (la receta).
- **Task** = un paso de esa lista (un ingrediente de la receta).
- **Trigger** = apretar "play" para que se ejecute.

Y ya. Con esto arrancamos.

---
# 📄 DAG 1: `mi_primer_dag` — el más simple

Este DAG hace **dos cositas**: primero escribe la fecha, después saluda. Nada más. Sirve para ver cómo se arma un DAG.

Vamos a leerlo entero y después parte por parte.

In [1]:
with open("./airflow-docker/dags/mi_primer_dag.py") as f:
    print(f.read())


from datetime import datetime
from airflow import DAG
from airflow.operators.bash import BashOperator
from airflow.operators.python import PythonOperator


def saludar():
    print("¡Hola desde mi primer DAG en Airflow! 👋")


with DAG(
    dag_id="mi_primer_dag",
    start_date=datetime(2024, 1, 1),
    schedule=None,          # None = solo se dispara manualmente
    catchup=False,
    tags=["ejemplo", "anyoneai"],
) as dag:

    tarea_bash = BashOperator(
        task_id="decir_fecha",
        bash_command="echo La fecha de hoy es: $(date)",
    )

    tarea_python = PythonOperator(
        task_id="saludar",
        python_callable=saludar,
    )

    tarea_bash >> tarea_python   # define el orden: primero bash, luego python



### Parte 1 — Los `import` (traer las herramientas)
```python
from datetime import datetime
from airflow import DAG
from airflow.operators.bash import BashOperator
from airflow.operators.python import PythonOperator
```
Esto es como sacar las herramientas de la caja antes de empezar. Le decimos a Python: *"voy a usar fechas, voy a usar DAG, y dos tipos de tareas: una que corre comandos (Bash) y otra que corre funciones de Python"*. No hace nada todavía, solo prepara.

### Parte 2 — La función que saluda
```python
def saludar():
    print("¡Hola desde mi primer DAG en Airflow! 👋")
```
Una función de Python normal y corriente. Solo imprime un saludo. Más abajo, una task la va a usar.

### Parte 3 — Crear el DAG (la receta)
```python
with DAG(
    dag_id="mi_primer_dag",              # el nombre que ves en la web
    start_date=datetime(2024, 1, 1),     # desde cuándo existe (poné una fecha vieja y listo)
    schedule=None,                       # None = NO corre solo, corre cuando vos apretás play
    catchup=False,                       # no intentes recuperar corridas viejas
    tags=["ejemplo", "anyoneai"],        # etiquetas para encontrarlo en la web
) as dag:
```
Acá creamos el DAG y le damos sus datos. Los dos que más importan:

- **`schedule=None`** → "este DAG NO se corre solo". Solo corre cuando vos le das play en la web. (Si un día querés que corra todos los días, se cambia por `"@daily"`.)
- **`catchup=False`** → "no te pongas al día con el pasado". 

> 💡 **¿Qué es `catchup`?** (lo tenías seleccionado). Imaginá que un DAG está configurado para correr todos los días desde enero, pero recién lo activás en marzo. Con `catchup=True`, Airflow intentaría correr **todos los días de enero, febrero y marzo de golpe** para "ponerse al día". Con `catchup=False`, arranca de hoy en adelante y listo. **Casi siempre querés `False`.**

### Parte 4 — Las dos tareas (los pasos)
```python
    tarea_bash = BashOperator(
        task_id="decir_fecha",
        bash_command="echo La fecha de hoy es: $(date)",
    )

    tarea_python = PythonOperator(
        task_id="saludar",
        python_callable=saludar,
    )
```
Dos pasos:
- **`tarea_bash`** corre un comando de terminal (`echo ...`) que escribe la fecha. `task_id` es el nombre del paso.
- **`tarea_python`** corre la función `saludar` que vimos arriba. `python_callable=saludar` significa "la función a ejecutar es `saludar`".

> Fijate que `BashOperator` y `PythonOperator` son solo **el tipo de cada paso**: uno corre comandos, el otro corre Python.

### Parte 5 — El orden
```python
    tarea_bash >> tarea_python   # primero bash, luego python
```
Esta única línea dice **el orden**: primero corre `tarea_bash`, y cuando termina, corre `tarea_python`.

La flecha `>>` se lee **"y después"**. `tarea_bash >> tarea_python` = "la tarea bash, y después la tarea python".

```
  decir_fecha  ──▶  saludar
```

**Y eso es todo el DAG.** Trae herramientas, define una función, crea el DAG, arma dos pasos, y dice en qué orden van. 🎉

---
# 📄 DAG 2: `mini_elt_taskflow_api` — uno un poquito más real

Este hace algo útil de verdad, en **3 pasos**:

```
  1. BAJAR      2. LIMPIAR        3. GUARDAR
  feriados  ─▶  los datos    ─▶   en un archivo
  de una web    (ordenarlos)      (una tablita)
```

Está escrito con **otro estilo** (con `@` en vez de `with`), pero hace lo mismo que aprendimos: pasos en orden. Leámoslo:

In [2]:
with open("./airflow-docker/dags/mini_elt_taskflow_api.py") as f:
    print(f.read())

"""
mini_elt_taskflow_api.py

Mini pipeline ELT de ejemplo para enseñar Apache Airflow con la **TaskFlow API**.

Hace un ELT completo en 3 pasos, usando la MISMA API de feriados que el proyecto
del Sprint 01 (date.nager.at):

    EXTRACT  -> pide los feriados de Brasil a la API (HTTP GET)
    TRANSFORM-> limpia los datos (selecciona columnas, parsea la fecha)
    LOAD     -> guarda el resultado en una tabla SQLite

Cómo probarlo:
    1. Copiá este archivo a la carpeta `dags/` de tu Airflow.
    2. Reiniciá el contenedor / scheduler para que lo detecte.
    3. Entrá a la GUI (localhost:8080), buscá el DAG "mini_elt_taskflow_api",
       activá el toggle y tocá ▶ (Trigger DAG).
    4. Mirá el grafo y los logs de cada task.

Requisitos dentro del entorno de Airflow: requests, pandas (sqlite3 viene con Python).
"""

from __future__ import annotations

import sqlite3
from datetime import datetime

import pandas as pd
import requests

from airflow.decorators import dag, task

# --- Configura

### El estilo con `@` (arroba), en simple
```python
@dag(dag_id="mini_elt_taskflow_api", schedule=None, catchup=False, ...)
def mini_elt_taskflow_api():
    ...
```
El `@dag` arriba de una función significa **"esta función es un DAG"**. Es otra forma de escribir lo mismo que el `with DAG(...)` del ejemplo anterior. Los datos son iguales: `schedule=None` (no corre solo) y `catchup=False` (no recupera el pasado).

> No te compliques con cuál estilo es mejor. **Los dos arman un DAG.** Este usa `@`, el otro usa `with`. Punto.

### Los 3 pasos (cada uno con `@task`)
El `@task` arriba de una función significa **"esta función es un paso del DAG"**.

**Paso 1 — bajar los datos:**
```python
@task()
def extract():
    response = requests.get(url)     # le pide los feriados a una web
    return response.json()           # devuelve la lista de feriados
```

**Paso 2 — limpiar:**
```python
@task()
def transform(raw):
    # se queda con 3 columnas y ordena la fecha
    return datos_limpios
```

**Paso 3 — guardar:**
```python
@task()
def load(rows):
    # guarda los datos en un archivo (una tablita)
    return cantidad_de_filas
```

Cada `@task` es un paso, igual que `tarea_bash` y `tarea_python` eran pasos en el otro DAG. Solo que acá se escriben como funciones con `@task` arriba.

### El orden, en este estilo
```python
    raw_data   = extract()            # paso 1
    clean_data = transform(raw_data)  # paso 2 (usa lo que dio el paso 1)
    load(clean_data)                  # paso 3 (usa lo que dio el paso 2)
```
Acá **no hay flechas `>>`**. El orden sale solo de cómo se llaman las funciones: `transform` usa el resultado de `extract`, entonces Airflow entiende que `extract` va **antes**. Y `load` usa lo de `transform`, así que va al final.

```
  extract  ──▶  transform  ──▶  load
```

El resultado de un paso viaja al siguiente automáticamente. Vos solo escribís Python normal.

> Y la última línea del archivo, `mini_elt_taskflow_api()`, es solo "prender" el DAG para que Airflow lo vea. En el estilo con `@` hay que ponerla; en el estilo con `with` no hace falta.

---
# ▶️ ¿Cómo los corro y veo que funcionan?

1. Abrí **http://localhost:8080** en el navegador (usuario `airflow`, contraseña `airflow`).
2. Buscá el DAG por su nombre en la lista.
3. Prendé el **interruptor** de la izquierda (de gris a azul).
4. Apretá el botón **▶️ (play)** para correrlo.
5. Entrá al DAG: vas a ver los pasos ponerse **verdes** cuando terminan bien (o rojos si fallan).
6. Clic en un paso → **Logs**: ahí ves lo que imprimió (los `print`).

> 🎯 **Regla mental final:** un DAG es una **lista de pasos en orden**. Airflow los corre, los vigila, y te muestra en verde/rojo cómo salieron. Todo lo demás son detalles que vas a ir aprendiendo de a poco.

## Resumen de una pantalla 📌

| Palabra | Qué es, fácil |
|---|---|
| **DAG** | Tu lista de pasos (un archivo `.py`). |
| **Task** | Un paso de la lista. |
| **`>>`** | "y después" — define el orden (estilo `with`). |
| **`@dag` / `@task`** | La otra forma de escribir un DAG y sus pasos (con arroba). |
| **`schedule=None`** | No corre solo; corre cuando apretás play. |
| **`catchup=False`** | No recupera corridas viejas del pasado. |
| **Trigger / ▶️** | Apretar play para correrlo. |
| **Verde / Rojo** | El paso salió bien / falló. |

**Tus dos DAGs hacen lo mismo en el fondo:** pasos en orden. Uno es más simple (`mi_primer_dag`), el otro hace algo más útil (`mini_elt`). Y están escritos en los dos estilos que existen. Con eso ya entendés Airflow. 💪